In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint # Import randint for RandomizedSearchCV

# dataset import
from google.colab import files

uploaded = files.upload()
for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')
  uploaded_filename = fn
df = pd.read_csv(uploaded_filename)

# EDA

print("-- DataFrame Head ---")
display(df.head())
print("\n--- DataFrame Info ---")
df.info()
print("\n--- DataFrame Description ---")
display(df.describe())

# distribution graphs
if 'fraud' in df.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(x='fraud', data=df)
    plt.title('Distribution of Fraud vs. Non-Fraud')
    plt.xlabel('Fraud (0: No Fraud, 1: Fraud)')
    plt.ylabel('Count')
    plt.show()


    # class distribution -- it's evenly distributed
    print("\n--- Class Distribution of 'fraud' column ---")
    class_counts = df['fraud'].value_counts()
    class_proportions = df['fraud'].value_counts(normalize=True)
    print("Class Counts:\n", class_counts)
    print("\nClass Proportions:\n", class_proportions)

# histogram comparison
print("\n--- Histograms for Numerical Features (excluding transaction_id) ---")
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

if 'transaction_id' in numerical_cols:
    numerical_cols.remove('transaction_id')
if 'fraud' in numerical_cols:
    numerical_cols.remove('fraud')

for col in numerical_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# scatterplot comparison
if 'amount' in df.columns and 'avg_transaction_value' in df.columns and 'fraud' in df.columns:
    print("\n--- Scatterplot: Amount vs. Avg Transaction Value (colored by Fraud) ---")
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x='amount', y='avg_transaction_value', hue='fraud', data=df, palette='viridis', alpha=0.7)
    plt.title('Transaction Amount vs. Average Transaction Value by Fraud Status')
    plt.xlabel('Transaction Amount')
    plt.ylabel('Average Transaction Value')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# correlation matrix
print("\n--- Correlation Matrix of Numerical Features ---")
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# feature engineering with transaction velocity and risk score
df['transaction_velocity'] = df['transaction_count_last_24h'] / (df['hour_of_day'] + 1e-6)

df['risk_score'] = df['amount'] * (1 + df['is_new_merchant'] * 0.5) # Example: 50% higher risk for new merchants

print("-- DataFrame with New Features ---")
display(df[['amount', 'hour_of_day', 'transaction_count_last_24h', 'is_new_merchant', 'transaction_velocity', 'risk_score', 'fraud']].head())

# SMOTE
X = df.drop(columns=['fraud', 'transaction_id'])
y = df['fraud']

print("Original class distribution:")
print(y.value_counts())

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("\nResampled class distribution:")
print(y_resampled.value_counts())

# Display the head of the resampled features (optional, for verification)
display(X_resampled.head())

# training and testing
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42, stratify=y_resampled)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train class distribution:\n{y_train.value_counts(normalize=True)}")
print(f"y_test class distribution:\n{y_test.value_counts(normalize=True)}")

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFirst 5 rows of scaled X_train:")
display(pd.DataFrame(X_train_scaled, columns=X.columns).head())

def evaluate_model(name, model, X_test, y_test):
    """Helper function to train and evaluate a model."""
    print(f"\n--- Evaluating {name} ---")
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    if y_pred_proba is not None:
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        print(f"ROC AUC Score: {roc_auc:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


# 1. Logistic Regression (Baseline)
log_reg = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets and binary classification
log_reg.fit(X_train_scaled, y_train)
evaluate_model("Logistic Regression", log_reg, X_test_scaled, y_test)

# 2. Decision Tree Classifier
dec_tree = DecisionTreeClassifier(random_state=42)
dec_tree.fit(X_train_scaled, y_train)
evaluate_model("Decision Tree", dec_tree, X_test_scaled, y_test)

# 3. Random Forest Classifier
rand_forest = RandomForestClassifier(random_state=42, n_estimators=100) # n_estimators is the number of trees
rand_forest.fit(X_train_scaled, y_train)
evaluate_model("Random Forest", rand_forest, X_test_scaled, y_test)

# 4. Gradient Boosting Classifier
grad_boost = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)
grad_boost.fit(X_train_scaled, y_train)
evaluate_model("Gradient Boosting", grad_boost, X_test_scaled, y_test)

# RandomizedSearchCV
param_distributions = {
    'n_estimators': randint(low=50, high=151), # Sample integers from 50 to 150
    'max_depth': [None, 10, 20], # Can still be a list of categories
    'min_samples_split': randint(low=2, high=11), # Sample integers from 2 to 10
    'min_samples_leaf': randint(low=1, high=5) # Sample integers from 1 to 4
}

# cv=3 for 3-fold cross-validation, scoring='roc_auc' because it's a good metric for imbalanced data
grid_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), # Use the RandomForestClassifier
    param_distributions=param_distributions, # Use param_distributions for RandomizedSearchCV
    n_iter=10, # Number of parameter settings that are sampled
    cv=3,
    scoring='roc_auc', # Evaluate based on AUC score
    n_jobs=-1, # Use all available CPU cores
    verbose=1 # Display progress
)

# Fit GridSearchCV to the training data
print("\n--- Performing GridSearchCV for Random Forest ---")
grid_search.fit(X_train_scaled, y_train)

# Print the best parameters and the best score
print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best cross-validation ROC AUC score: {grid_search.best_score_:.4f}")

# Get the best model
best_rf_model = grid_search.best_estimator_

# Evaluate the best model on the test set
evaluate_model("Tuned Random Forest", best_rf_model, X_test_scaled, y_test)

# feature importance
print("\n--- Feature Importance from Tuned Random Forest Model ---")
feature_importances = pd.Series(best_rf_model.feature_importances_, index=X.columns)
top_3_features = feature_importances.nlargest(3)
print("Top 3 Fraud-Driving Features (based on Random Forest Feature Importance):")
for i, (feature, importance) in enumerate(top_3_features.items()):
    print(f"{i+1}. {feature}: {importance:.4f}")

# risk
def assign_fraud_score_and_category(model, scaler, transaction_df, thresholds={'LOW': 0.3, 'MEDIUM': 0.7}):
    """
    Calculates fraud probability and assigns a risk category for each transaction.

    Args:
        model: The trained machine learning model (e.g., best_rf_model).
        scaler: The fitted StandardScaler.
        transaction_df: A pandas DataFrame containing new transactions (features only).
                        Assumes transaction_df has the same columns as the training data used for scaling.
        thresholds: A dictionary defining the probability thresholds for risk categories.
                    Keys are category names (e.g., 'LOW', 'MEDIUM'), values are the upper bounds
                    for the probability to fall into that category. 'HIGH' is typically above the last threshold.

    Returns:
        A pandas DataFrame with 'fraud_probability' and 'risk_category' columns.
    """
    # Scale the input features
    transaction_scaled = scaler.transform(transaction_df)

    # Predict fraud probabilities
    fraud_probabilities = model.predict_proba(transaction_scaled)[:, 1]

    # Assign risk categories based on probabilities and thresholds
    risk_categories = []
    for prob in fraud_probabilities:
        if prob < thresholds['LOW']:
            risk_categories.append('LOW')
        elif prob < thresholds['MEDIUM']:
            risk_categories.append('MEDIUM')
        else:
            risk_categories.append('HIGH')

    # Create a DataFrame for the results
    results_df = pd.DataFrame({
        'fraud_probability': fraud_probabilities,
        'risk_category': risk_categories
    }, index=transaction_df.index)

    return results_df


print("\n--- Applying Fraud Scoring Function to Test Data ---")
# Apply the fraud scoring function to the original unscaled test set (X_test)
# The `assign_fraud_score_and_category` function handles scaling internally.
fraud_scoring_results = assign_fraud_score_and_category(best_rf_model, scaler, X_test, thresholds={'LOW': 0.3, 'MEDIUM': 0.7})

print("First 5 rows of Fraud Scoring Results:")
display(fraud_scoring_results.head())

print("\n--- Distribution of Risk Categories in Test Data ---")
risk_category_counts = fraud_scoring_results['risk_category'].value_counts().reindex(['LOW', 'MEDIUM', 'HIGH'], fill_value=0)
print(risk_category_counts)

# Visualize the distribution of risk categories
plt.figure(figsize=(8, 6))
sns.countplot(x='risk_category', data=fraud_scoring_results, order=['LOW', 'MEDIUM', 'HIGH'], palette='viridis')
plt.title('Distribution of Fraud Risk Categories in Test Data')
plt.xlabel('Risk Category')
plt.ylabel('Number of Transactions')
plt.show()
